# EX03: AST Traversals with `Lark`

## Grammar Engineering

## MEI/2026-27

#### Nuno Macedo
Universidade do Minho

The following exercises are intended to be solved using a parser generator module for Python, [Lark](https://lark-parser.readthedocs.io/).

# Module [`Lark`](https://lark-parser.readthedocs.io/) basics

- In `Lark`, the grammar is purely syntactic, there are no semantic actions assigned

- It automatically generates an AST and provides methods to traverse it

- By decoupling the parsing and the transformation, it is more prone to multiple analysis

- No clear distinction between the lexer and the parser, defined in the same specification

- Richer range of parsing algorithms, supports more grammars

In [1]:
!pip install lark

# `Lark` [`Transformer`](https://lark-parser.readthedocs.io/en/stable/visitors.html#transformer)

- Processes the tree **bottom-up**: children are transformed before their parent

- Each rule method returns a replacement — producing a new tree or synthesized values

- Ideal for AST construction, translation, and simplification (pure functional style)

- Best when transformation is local and context-free (no need for parent info)

## Example: Evaluate simple arithmetic expressions

- A transformer can be used to evaluate a tree bottom-up: each node receives the trasformed value of the children, synthesizes values, and propagates them up

In [2]:
from lark import Lark, Transformer

grammar_simple_arithmetic = r"""
?start: expr

?expr : term ((ADD|SUB) term)*

?term : factor ((MUL|DIV) factor)*

?factor : SUB? literal

?literal : NUMBER | "(" expr ")"

ADD: "+"
SUB: "-"
MUL: "*"
DIV: "/"

NUMBER: /[0-9]+(\.[0-9]+)?/

%ignore /[ \t\f]+/
"""

class EvalTransformer(Transformer):
    def expr(self, items):
        res = items[0]
        for i in range(1, len(items), 2):
          if items[i].type == 'ADD':
            res += items[i+1]
          elif items[i].type == 'SUB':
            res -= items[i+1]
        return res

    def term(self, items):
        res = items[0]
        for i in range(1, len(items), 2):
          if items[i].type == 'MUL':
            res *= items[i+1]
          elif items[i].type == 'DIV':
            res /= items[i+1]
        return res

    def factor(self, items):
        return -items[1]

    def NUMBER(self, token):
        return float(token)

In [3]:
parser_simple_arithmetic = Lark(grammar_simple_arithmetic)

print(EvalTransformer().transform(parser_simple_arithmetic.parse("(-3.4 * 30) + 20 + -30 * 100")))
print(EvalTransformer().transform(parser_simple_arithmetic.parse("(10 + 30 + 100) * (1 - 1)")))

-3082.0
0.0


## Example: Flatten arithmetic expressions with declared variables

- We can use the transformers to modify the AST

- Passing information top-down requires shared state

- Create a symbol table and update it accordingly

- Traversal is left-to-right, bottom-down

- Must distinguish between `ID` declarations and `ID` calls

- Notice the use of EBNF and nodes with arbitrary number of children

In [4]:
from lark import Lark, Transformer
from lark.visitors import VisitError

grammar_decls_arithmetic = r"""
?start : decls expr

decls : decl*

decl : "let" id_decl "=" expr ";"

id_decl : ID

?expr : term ((ADD|SUB) term)*

?term : factor ((MUL|DIV) factor)*

?factor : SUB? literal

?literal : id_call | NUMBER | "(" expr ")"

id_call : ID

ADD: "+"
SUB: "-"
MUL: "*"
DIV: "/"

ID : /[a-z]+/
NUMBER: /[0-9]+(\.[0-9]+)?/

%ignore /[ \t]+/
"""

class FlattenTransformer(Transformer):
    def __init__(self):
      self.env = {}

    def start(self, items):
      return items[1]

    def id_decl(self, items):
      return items[0]

    def decl(self, items):
      self.env[items[0].value] = items[1]

    def id_call(self, items):
      if items[0].value not in self.env:
        raise ValueError("Undeclared variable " + items[0].value)
      return self.env[items[0].value]

In [5]:
parser_decls_arithmetic = Lark(grammar_decls_arithmetic)

for example in ["let x = 10; 10 + x",
                "let x = 10; let y = x * 4; 10 * y + x",
                "let x = 10; 10 * y + x",
                "let x = x; 10 + x"]:
  try:
    print(FlattenTransformer().transform(parser_decls_arithmetic.parse(example)).pretty())
  except VisitError as e:
    print(f"Exception raised: {e.orig_exc}")

expr
  10
  +
  10

expr
  term
    10
    *
    term
      10
      *
      4
  +
  10

Exception raised: Undeclared variable y
Exception raised: Undeclared variable x


## Example: Evaluate arithmetic expressions with declared variables

### Version with two traversals

- After flattening, we can just use the previous transformer to evaluate

- Cleaner, but two traversals

In [6]:
print(EvalTransformer().transform(FlattenTransformer().transform(parser_decls_arithmetic.parse("let x = 10; let y = x * 4; 10 * y + x"))))

410.0


### Version with shared state

- Instead of flattening the declarations, evaluate values during traversal

In [7]:
class EvalDeclsTransformer(Transformer):
  def __init__(self):
    self.env = {}

  def start(self, items):
    return items[1]

  def id_decl(self, items):
    return items[0]

  def decl(self, items):
    self.env[items[0].value] = items[1]

  def expr(self, items):
      res = items[0]
      for i in range(1, len(items), 2):
        if items[i].type == 'ADD':
          res += items[i+1]
        elif items[i].type == 'SUB':
          res -= items[i+1]
      return res

  def term(self, items):
      res = items[0]
      for i in range(1, len(items), 2):
        if items[i].type == 'MUL':
          res *= items[i+1]
        elif items[i].type == 'DIV':
          res /= items[i+1]
      return res

  def factor(self, items):
      return -items[1]

  def NUMBER(self, token):
      return float(token)

  def id_call(self, items):
    if items[0].value not in self.env:
      raise ValueError("Undeclared variable " + items[0].value)
    return self.env[items[0].value]

In [8]:
for example in ["let x = 10; 10 + x",
                "let x = 10; let y = x * 4; 10 * y + x",
                "let x = 10; 10 * y + x",
                "let x = x; 10 + x"]:
  try:
    print(EvalDeclsTransformer().transform(parser_decls_arithmetic.parse(example)))
  except VisitError as e:
    print(f"Exception raised: {e.orig_exc}")

20.0
410.0
Exception raised: Undeclared variable y
Exception raised: Undeclared variable x


### Version with Python closures

- The synthesized values are functions waiting for an environment to evaluate

- Cleaner information flow, no shared state, parent nodes resolve pending evaluations

- Code more cluttered, must propagate function declarations

In [9]:
class EvalClosuresTransformer(Transformer):
    def start(self, items):
      return items[1](items[0])

    def decls(self, items):
      res = {}
      for i,j in items:
        res[i] = j(res)
      return res

    def id_decl(self, items):
      return items[0]

    def decl(self, items):
      return items[0].value, items[1]

    def expr(self, items):
      def eval(env):
        res = items[0](env)
        for i in range(1,len(items)-1,2):
          if items[i].type == "ADD":
            res += items[i+1](env)
          elif items[i].type == "SUB":
            res -= items[i+1](env)
        return res
      return eval

    def term(self, items):
      def eval(env):
        res = items[0](env)
        for i in range(1,len(items)-1,2):
          if items[i].type == "DIV":
            res /= items[i+1](env)
          elif items[i].type == "MUL":
            res *= items[i+1](env)
        return res
      return eval

    def literal(self, items):
      def eval(env):
        x = items[1]
        a = x(env)
        return -a
      return eval

    def id_call(self, items):
      def eval(env):
        x = items[0].value
        if x not in env:
          raise ValueError("Undeclared variable " + x)
        return env[x]
      return eval

    def NUMBER(self, items):
      def eval(env):
        return float(items)
      return eval

In [10]:
for example in ["let x = 10; 10 + x",
                "let x = 10; let y = x * 4; 10 * y + x",
                "let x = 10; 10 * y + x",
                "let x = x; 10 + x"]:
  try:
    print(EvalClosuresTransformer().transform(parser_decls_arithmetic.parse(example)))
  except VisitError as e:
    print(f"Exception raised: {e.orig_exc}")

20.0
410.0
Exception raised: Undeclared variable y
Exception raised: Undeclared variable x


# `Lark` [`Visitor`](https://lark-parser.readthedocs.io/en/stable/visitors.html#visitor)

- Processes the tree **bottom-up**, but does not replace nodes — used for analysis or annotation

- Methods are called after children are visited, enabling information gathering

- Useful for semantic checks, statistics, or data collection from the tree

- Side-effect-oriented, emphasizes read-only inspection rather than transformation

## Example: Collect declarations and variables

- A visitor can simply retrieve variables and add them to a global state

In [11]:
from lark import Lark, Visitor

class CollectorVisitor(Visitor):
    def __init__(self):
      self.decs = set()
      self.vars = set()

    def id_decl(self, items):
      self.decs.add(items.children[0].value)

    def id_call(self, items):
      self.vars.add(items.children[0].value)

In [12]:
for example in ["let x = 10; 10 + x",
                "let x = 10; let y = x * 4; 10 * y + x",
                "let x = 10; 10 * y + x",
                "let x = x; 10 + x"]:
  try:
    c = CollectorVisitor()
    c.visit(parser_decls_arithmetic.parse(example))
    print(f"Decls: {c.decs}, Vars: {c.vars}")
  except VisitError as e:
    print(f"Exception raised: {e.orig_exc}")

Decls: {'x'}, Vars: {'x'}
Decls: {'y', 'x'}, Vars: {'y', 'x'}
Decls: {'x'}, Vars: {'y', 'x'}
Decls: {'x'}, Vars: {'x'}


## Example: Rename all identifiers

- A visitor can simply change the values of all ID tokens

- *Note:* `pretty()` does not reflect changes to tokens

In [13]:
from lark import Lark, Visitor

class RenameVisitor(Visitor):

    def id_decl(self, items):
      items.children[0].value = items.children[0].value.upper()

    def id_call(self, items):
      items.children[0].value = items.children[0].value.upper()

In [14]:
c = RenameVisitor()
t = parser_decls_arithmetic.parse("let x = 10; let y = x * 4; 10 * y + x")
c.visit(t)
print(t)

Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'decls'), [Tree(Token('RULE', 'decl'), [Tree(Token('RULE', 'id_decl'), [Token('ID', 'X')]), Token('NUMBER', '10')]), Tree(Token('RULE', 'decl'), [Tree(Token('RULE', 'id_decl'), [Token('ID', 'Y')]), Tree(Token('RULE', 'term'), [Tree(Token('RULE', 'id_call'), [Token('ID', 'X')]), Token('MUL', '*'), Token('NUMBER', '4')])])]), Tree(Token('RULE', 'expr'), [Tree(Token('RULE', 'term'), [Token('NUMBER', '10'), Token('MUL', '*'), Tree(Token('RULE', 'id_call'), [Token('ID', 'Y')])]), Token('ADD', '+'), Tree(Token('RULE', 'id_call'), [Token('ID', 'X')])])])


# `Lark` [`Interpreter`](https://lark-parser.readthedocs.io/en/stable/visitors.html#interpreter)

- Walks the tree **top-down**, and you must explicitly call `visit()` on children

- Like visitors, does not create a new tree

- Provides full control over traversal order and context propagation

- Good for conditional execution, short-circuiting logic, or partial evaluation

# Exercise 1

Recall the language for intervals from Exercises 2 and 4 of [EX02](https://colab.research.google.com/drive/1NLEBOm8PxmG52P0Rhjcu3oIFGtAF90Sw). Consider now that the list of intervals is optionally preceded by a `+` or `-` sign that determines whether all intervals should be increasing or decreasing, respectively.

## Exercise 1.1

Write a `Lark` grammar to obtain the AST for interval ranges with sign annotations.

In [15]:
from lark import Lark, Transformer

def run_parser(data):
  """
  Lark parser with doctests.

  >>> run_parser('+[-1.5 ; +2] [3 ; +4.7]')
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'sign'), [Token('PLUS', '+')]), Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('FLOAT', '-1.5')]), Tree(Token('RULE', 'number'), [Token('INT', '+2')])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '3')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '+4.7')])])])])

  >>> run_parser("-[3 ; 3.5; 4] [5 ; 12] [48 ; 73.2]")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'sign'), [Token('MINUS', '-')]), Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '3')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '3.5')]), Tree(Token('RULE', 'number'), [Token('INT', '4')])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '5')]), Tree(Token('RULE', 'number'), [Token('INT', '12')])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '48')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '73.2')])])])])

  >>> run_parser("[3 ; 3.5] [1 ; 10; 12] [48 ; 73.2]")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '3')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '3.5')])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '1')]), Tree(Token('RULE', 'number'), [Token('INT', '10')]), Tree(Token('RULE', 'number'), [Token('INT', '12')])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '48')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '73.2')])])])])

  >>> run_parser("+[3 ; 3.5] [2 ; 1] [0.5 ; -73.2]")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'sign'), [Token('PLUS', '+')]), Tree(Token('RULE', 'intervals'), [Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '3')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '3.5')])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('INT', '2')]), Tree(Token('RULE', 'number'), [Token('INT', '1')])]), Tree(Token('RULE', 'interval'), [Tree(Token('RULE', 'number'), [Token('FLOAT', '0.5')]), Tree(Token('RULE', 'number'), [Token('FLOAT', '-73.2')])])])])
  """
  grammar = r"""
  start: sign? intervals

  sign: PLUS | MINUS
  intervals: interval+
  interval: "[" number (";" number)+ "]"
  number: INT | FLOAT

  PLUS: "+"
  MINUS: "-"
  
  FLOAT: /[+-]?\d+\.\d+/
  INT: /[+-]?\d+/

  %ignore /[ \t\n\f]+/
  """

  parser = Lark(grammar)
  return parser.parse(data)

import doctest
doctest.run_docstring_examples(run_parser, globals())

## Exercise 1.2

Write a `Lark` transformer to validate the intervals (always increasing or always decreasing, according to the prefix sign).

In [ ]:
def validate(data):
  """
  Lark interval validator with doctests.

  >>> print(validate('-[11 ; 0] [0 ; -1; -1.5] [-2 ; -3] [-3 ; -4]'))
  True
  >>> print(validate('-[-200 ; -140; -123] [-100 ; 0] [-1.5 ; +2] [3 ; +4.7]'))
  False
  >>> print(validate("[3 ; 3.5] [5 ; 12] [48 ; 13.2]"))
  True
  >>> print(validate("+[34 ; 36.5]"))
  True
  >>> print(validate("+[3 ; 3.5] [63 ; 80; 12] [98 ; 173.2]"))
  False
  >>> print(validate("[3 ; 3.5] [2 ; 1] [0.5 ; -73.2]"))
  True
  >>> print(validate('+[0 ; 5; 10] [5 ; 10]'))
  True
  """
  class ValidationTransformer(Transformer):
    pass

  return ValidationTransformer().transform(run_parser(data))

import doctest
doctest.run_docstring_examples(validate, globals())

# Exercise 2

Recall Exercises 1 and 3 from [EX02](https://colab.research.google.com/drive/1NLEBOm8PxmG52P0Rhjcu3oIFGtAF90Sw) to process propositional logic formulas. We want to extend the logic to allow the encoding of *first-order logic formulas*, introducing the notion of quantifier and predicate of arity $n$.

First-order logic formulas have thus the following syntax:

- Predicates $A$, $B$, $C$, ...
- Variables $a$, $b$, $c$, ...
- Constants false $⊥$ and true $⊤$
- Predicate application $A(a_1, \ldots, a_n)$ if $A$ has arity $n$
- Conjunction $f_1 ∧ f_2$,
- Disjunction $f_1 ∨ f_2$,
- Negation $¬ f_1$,
- Implication $f_1 → f_2$
- Equivalence $f_1 ↔︎ f_2$
- Universal quantification $∀ a . f_1$
- Existential quantification $∃ a . f_1$

Variables are introduced by quantifications, while predicates must be declared externally and have their arity assigned. For instance, the following is a valid first-order formula declaration:
```
A : 1
R : 2
∀ a . A(a) → ∃ b . R(a,b)
```

## Exercise 2.1

Write an attribute grammar (AG) for this language:
- Write the context-free grammar
- Identifying the synthesized and inherited attributes for each grammar variable
- Define the evaluation rules for each attribute

## Exercise 2.2

Write a `Lark` grammar to obtain the AST for first-order logic formula declarations.

In [ ]:
from lark import Lark, Transformer

def run_parser(data):
  """
  Lark parser with doctests.

  >>> run_parser("A : 0 B: 0 A() ∧ ¬B()")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'decls'), [Tree(Token('RULE', 'decl'), [Token('PRD', 'A'), Token('INT', '0')]), Tree(Token('RULE', 'decl'), [Token('PRD', 'B'), Token('INT', '0')])]), Tree(Token('RULE', 'conj'), [Tree(Token('RULE', 'application'), [Token('PRD', 'A'), Tree(Token('RULE', 'vars'), [])]), Token('CONJ', '∧'), Tree(Token('RULE', 'neg'), [Token('NEG', '¬'), Tree(Token('RULE', 'application'), [Token('PRD', 'B'), Tree(Token('RULE', 'vars'), [])])])])])

  >>> run_parser("A : 1 ∃ b . A(b)")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'decls'), [Tree(Token('RULE', 'decl'), [Token('PRD', 'A'), Token('INT', '1')])]), Tree(Token('RULE', 'formula'), [Token('SOME', '∃'), Tree(Token('RULE', 'bnd'), [Token('ID', 'b')]), Tree(Token('RULE', 'application'), [Token('PRD', 'A'), Tree(Token('RULE', 'vars'), [Token('ID', 'b')])])])])

  >>> run_parser("A : 1 R : 2 ∀ a . (A(a) → ∃ b . R(a,b))")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'decls'), [Tree(Token('RULE', 'decl'), [Token('PRD', 'A'), Token('INT', '1')]), Tree(Token('RULE', 'decl'), [Token('PRD', 'R'), Token('INT', '2')])]), Tree(Token('RULE', 'formula'), [Token('ALL', '∀'), Tree(Token('RULE', 'bnd'), [Token('ID', 'a')]), Tree(Token('RULE', 'implies'), [Tree(Token('RULE', 'application'), [Token('PRD', 'A'), Tree(Token('RULE', 'vars'), [Token('ID', 'a')])]), Token('IMPLY', '→'), Tree(Token('RULE', 'formula'), [Token('SOME', '∃'), Tree(Token('RULE', 'bnd'), [Token('ID', 'b')]), Tree(Token('RULE', 'application'), [Token('PRD', 'R'), Tree(Token('RULE', 'vars'), [Token('ID', 'a'), Token('ID', 'b')])])])])])])

  >>> run_parser("⊤")
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'decls'), []), Tree(Token('RULE', 'constant'), [Token('TRUE', '⊤')])])
  """
  grammar = r"""

  """

  parser = Lark(grammar)
  #print(parser.parse(data).pretty())
  return parser.parse(data)

import doctest
doctest.run_docstring_examples(run_parser, globals())

## Exercise 2.3

Write a `Lark` transformer to validate the declaration of first-order logic formulas. Namely:
- A variable $a$ may only occur if it has been introduced by a quantifier
- A predicate $A$ may only occur if it has been introduced by a quantifier
- A predicate application $A(a_1,\ldots,a_n)$ may conform to the arity declared for $A$

In [ ]:
def validate(data):
  """
  Lark validator with doctests.

  >>> print(validate("A : 0 B: 0 A() ∧ ¬B()"))
  None

  >>> print(validate("A : 0 B: 0 X() ∧ ¬B()"))
  Undeclared predicate called X

  >>> print(validate("A : 1 B: 0 A() ∧ ¬B()"))
  Invalid arity in call to A

  >>> print(validate("A : 1 B: 0 A(x) ∧ ¬B()"))
  Undeclared variable x

  >>> print(validate("A : 1 B: 0 ∀ x . A(x) ∧ ¬B()"))
  None

  >>> print(validate("A : 1 R : 2 ∀ a . (A(a) → ∃ b . R(a,b))"))
  None

  >>> print(validate("A : 1 R : 2 ∀ a . (A(a) → ∃ b . R(a,b))"))
  None

  >>> print(validate("A : 1 R : 2 ∀ a . (A(a) → ∃ b . R(a,b,c))"))
  Undeclared variable c

  >>> print(validate("A : 1 R : 2 ∀ a . (A(a) → ∃ b . R(a,b,a))"))
  Invalid arity in call to R

  >>> print(validate("A : 1 R : 2 ∀ a . (A(a) → ∃ b . R(b))"))
  Invalid arity in call to R

  >>> print(validate("⊤"))
  None

  >>> print(validate("A : 1 (∀ a . A(a)) ∧ A(a)"))
  Undeclared variable a
  """
  class ValidateTransformer(Transformer):
    pass

  return ValidateTransformer().transform(run_parser(data))

import doctest
doctest.run_docstring_examples(validate, globals())

# Exercise 3

Recall the tiny functional language from Exercise 5 of [EX02](https://colab.research.google.com/drive/1NLEBOm8PxmG52P0Rhjcu3oIFGtAF90Sw). Now, extend its support to also allow variables, declared with a given type. So a formula can be:

- Integer and Boolean literals
- A previously declared variable
- Binary operators (`+`, `*`, `<`, `and`, `or`)
- Conditional expressions (`if ... then ... else ...`)

For instance, the following is a valid expression in this language:
```
x : int
y : bool
if 2 > 3 or y then 10 else -1*x
```

## Exercise 3.1

Write a `Lark` grammar to obtain the AST for this tiny language.

In [ ]:
from lark import Lark, Transformer, v_args

def run_parser(data):
  """
  Lark parser with doctests.

  >>> run_parser('x : int \\n if 2 > x or 4 < 9 then 10 else -1*10')
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'decl'), [Token('ID', 'x'), Tree(Token('RULE', 'int'), [])]), Tree(Token('RULE', 'ifthenelse'), [Tree(Token('RULE', 'disj'), [Tree(Token('RULE', 'comp'), [Token('INT', '2'), Token('GT', '>'), Tree(Token('RULE', 'var'), [Token('ID', 'x')])]), Token('OR', 'or'), Tree(Token('RULE', 'comp'), [Token('INT', '4'), Token('LT', '<'), Token('INT', '9')])]), Token('INT', '10'), Tree(Token('RULE', 'arithterm'), [Token('INT', '-1'), Token('MUL', '*'), Token('INT', '10')])])])

  >>> run_parser('x : int \\n if 2*2 > x or 4 < 9 then 10 else -1*10')
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'decl'), [Token('ID', 'x'), Tree(Token('RULE', 'int'), [])]), Tree(Token('RULE', 'ifthenelse'), [Tree(Token('RULE', 'disj'), [Tree(Token('RULE', 'comp'), [Tree(Token('RULE', 'arithterm'), [Token('INT', '2'), Token('MUL', '*'), Token('INT', '2')]), Token('GT', '>'), Tree(Token('RULE', 'var'), [Token('ID', 'x')])]), Token('OR', 'or'), Tree(Token('RULE', 'comp'), [Token('INT', '4'), Token('LT', '<'), Token('INT', '9')])]), Token('INT', '10'), Tree(Token('RULE', 'arithterm'), [Token('INT', '-1'), Token('MUL', '*'), Token('INT', '10')])])])

  >>> run_parser('x : int \\n p : bool \\n if 2 > 3 or p then 10 else -x*10')
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'decls'), [Tree(Token('RULE', 'decl'), [Token('ID', 'x'), Tree(Token('RULE', 'int'), [])]), Tree(Token('RULE', 'decl'), [Token('ID', 'p'), Tree(Token('RULE', 'bool'), [])])]), Tree(Token('RULE', 'ifthenelse'), [Tree(Token('RULE', 'disj'), [Tree(Token('RULE', 'comp'), [Token('INT', '2'), Token('GT', '>'), Token('INT', '3')]), Token('OR', 'or'), Tree(Token('RULE', 'var'), [Token('ID', 'p')])]), Token('INT', '10'), Tree(Token('RULE', 'arithterm'), [Tree(Token('RULE', 'literal'), [Token('SUB', '-'), Tree(Token('RULE', 'var'), [Token('ID', 'x')])]), Token('MUL', '*'), Token('INT', '10')])])])

  >>> run_parser('x : int \\n p : bool \\n if True or x and 10 < 20 then 10 else -1*10*30')
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'decls'), [Tree(Token('RULE', 'decl'), [Token('ID', 'x'), Tree(Token('RULE', 'int'), [])]), Tree(Token('RULE', 'decl'), [Token('ID', 'p'), Tree(Token('RULE', 'bool'), [])])]), Tree(Token('RULE', 'ifthenelse'), [Tree(Token('RULE', 'disj'), [Token('BOOL', 'True'), Token('OR', 'or'), Tree(Token('RULE', 'conj'), [Tree(Token('RULE', 'var'), [Token('ID', 'x')]), Token('AND', 'and'), Tree(Token('RULE', 'comp'), [Token('INT', '10'), Token('LT', '<'), Token('INT', '20')])])]), Token('INT', '10'), Tree(Token('RULE', 'arithterm'), [Token('INT', '-1'), Token('MUL', '*'), Token('INT', '10'), Token('MUL', '*'), Token('INT', '30')])])])

  >>> run_parser('x : int \\n p : bool \\n if p then if not p then 10 else x+20 else if p and 20 > 23 then x*x else 50')
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'decls'), [Tree(Token('RULE', 'decl'), [Token('ID', 'x'), Tree(Token('RULE', 'int'), [])]), Tree(Token('RULE', 'decl'), [Token('ID', 'p'), Tree(Token('RULE', 'bool'), [])])]), Tree(Token('RULE', 'ifthenelse'), [Tree(Token('RULE', 'var'), [Token('ID', 'p')]), Tree(Token('RULE', 'ifthenelse'), [Tree(Token('RULE', 'neg'), [Token('NOT', 'not'), Tree(Token('RULE', 'var'), [Token('ID', 'p')])]), Token('INT', '10'), Tree(Token('RULE', 'arithexpr'), [Tree(Token('RULE', 'var'), [Token('ID', 'x')]), Token('SUM', '+'), Token('INT', '20')])]), Tree(Token('RULE', 'ifthenelse'), [Tree(Token('RULE', 'conj'), [Tree(Token('RULE', 'var'), [Token('ID', 'p')]), Token('AND', 'and'), Tree(Token('RULE', 'comp'), [Token('INT', '20'), Token('GT', '>'), Token('INT', '23')])]), Tree(Token('RULE', 'arithterm'), [Tree(Token('RULE', 'var'), [Token('ID', 'x')]), Token('MUL', '*'), Tree(Token('RULE', 'var'), [Token('ID', 'x')])]), Token('INT', '50')])])])

  >>> run_parser('20')
  Tree(Token('RULE', 'start'), [Tree(Token('RULE', 'decls'), []), Token('INT', '20')])
  """
  grammar = r"""


  """
  parser = Lark(grammar)
  #print(parser.parse(data).pretty())
  return parser.parse(data)

import doctest
doctest.run_docstring_examples(run_parser, globals())

## Exercise 3.2

Write a `Lark` transformer to type check this language.

For instance, the following expression type checks:
```
x : int
p : bool
if 2 > x or p then 10 else -x*10
```

But the following (syntactically correct) should throw a type error:

```
p : bool
if 2 > p then 10 else -1*10
```
```
x : int
p : bool
if 2 > 3 or 4 < 10 then x else p
```
```
p : bool
if 2 > 3 and 3 + p then 10 < 20 else True
```

In [ ]:
def type_check(data):
  """
  Lark type checker with doctests.

  >>> print(type_check('if 2 > 3 or 4 < 9 then 10 else -1*10'))
  <class 'int'>

  >>> print(type_check('if 2 > 3 or 4 then 10 else -1*10'))
  Type error in or

  >>> print(type_check('if True or False and 10 < 20 then 10 else -1*10*30'))
  <class 'int'>

  >>> print(type_check('if True then 10 else -1*(10 and 20)'))
  Type error in and

  >>> print(type_check('if 2 > 3 then 10 else True'))
  Type error in if-then-else

  >>> print(type_check('if 2 > 3 then if True then 10 else 20 else 30'))
  <class 'int'>

  >>> print(type_check('if False then if True then 10 else 20 else if True then 20 else 50'))
  <class 'int'>

  >>> print(type_check('if 10 < 20 then 20 < 10 else 20 < 23'))
  <class 'bool'>
  """
  class TypeCheckerTransformer(Transformer):
      pass

  return TypeCheckerTransformer().transform(run_parser(data))

import doctest
doctest.run_docstring_examples(type_check, globals())

# Exercise 4

Consider now that the previous language was extended with the definition of functions, which can also contain declared variables. Consider the following rules:
- A function and a variable can only be called if declared previously
- The scope of variables declared inside functions is always local
- A function can redefine a variable
- A function can use variables from the global scope

For instance, the following is correct code:
```
let g = 20;
let f(x) = let y = x * 2; y * g;
f(20) + 10
```

```
let f(x) = let y = x * 2; y * 10;
let x = 2; f(x)+10
```

```
let x = 2;
let f(x) = let y = x * 2; y * 10;
f(x) + 10
```


But the following is invalid:
```
let f(x) = let y = x * 2; y * 10;
f(x) + 10
```

## Exercise 4.1

Write a `Lark` grammar to parse this language.

In [ ]:
from lark import Lark, Transformer, v_args

grammar = r"""


"""

parser = Lark(grammar)
print(parser.parse(r"""
let g = 20;
let f(x) = let y = x * 2; y * g;
let h(x,y) = x*y*h*f(x);
f(20) + 10;
""").pretty())

## Exercise 4.2

Use `Lark` AST traversals to check the scopes of the variables.

In [ ]:
from lark.visitors import Transformer, VisitError

class ScopesVisitor(Transformer):
  pass

parser = Lark(grammar)
t = parser.parse(r"""
let g = 20;
let f(x) = let y = x * 2; y * g;
let h(x,y) = x*y*h*f(x);
f(20) + 10;
""")
v = ScopesVisitor()
v.transform(t)
print(v.scopes)


## Exercise 4.3

Write a Lark `Visitor` to rename local calls with their scope name. For instance, a variable `x` inside a function `f`, should be renamed as `f_x`.

## Exercise 4.4

Write Lark `Transformer` to evaluate the value of expressions in this language.

-- nmm, 2026-10-15